# Huấn luyện CNN–LSTM song song theo từng dataset

Notebook này huấn luyện **ba mô hình độc lập** trên `kaggle`, `csic` và `obfuscation`.

- Không trộn dữ liệu giữa các nguồn.
- Mỗi dataset có split 72% train / 8% validation / 20% test với seed 42.
- Tokenizer char-level chỉ fit trên train split của chính dataset đó.
- Obfuscation được chia theo `original_pattern` để các biến thể của cùng pattern không rơi vào nhiều split.
- Mỗi dataset lưu model, tokenizer, cấu hình và metrics vào thư mục artifact riêng.
- Padding, LSTM sequence pooling và early stopping được đồng bộ với mô hình tuần tự để giảm yếu tố gây nhiễu khi so sánh topology.

Nhánh CNN và LSTM chạy song song từ cùng embedding rồi concatenate trước tầng phân loại.


## 1. Thiết lập môi trường và cấu hình


In [8]:
from pathlib import Path
import gc
import json
import os
import pickle
import random
import sys
import time

START_DIR = Path.cwd().resolve()
if (START_DIR / "preprocessing" / "preprocess_data.py").exists():
    PROJECT_ROOT = START_DIR
elif START_DIR.name == "cnn_lstm_parallel" and (START_DIR.parent / "preprocessing" / "preprocess_data.py").exists():
    PROJECT_ROOT = START_DIR.parent
else:
    raise FileNotFoundError("Hãy chạy notebook từ repo root hoặc thư mục cnn_lstm_parallel.")

NOTEBOOK_DIR = PROJECT_ROOT / "cnn_lstm_parallel"
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(NOTEBOOK_DIR)

import numpy as np
import pandas as pd
import tensorflow as tf
from IPython.display import display
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.layers import (
    Concatenate, Conv1D, Dense, Dropout, Embedding,
    GlobalMaxPooling1D, Input, LSTM, MaxPooling1D,
)
from tensorflow.keras.metrics import AUC, Precision, Recall
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.utils import Sequence

from preprocessing import preprocess_data as prep

SEED = 42
MAX_LEN = 1024
EMBEDDING_DIM = 64
BATCH_SIZE = 128
EPOCHS = 50
TEST_SIZE = 0.20
VAL_SIZE_WITHIN_TRAIN_VAL = 0.10  # 10% của 80% còn lại = 8% toàn bộ dữ liệu.
DECISION_THRESHOLD = 0.50

KAGGLE_PATH = PROJECT_ROOT / "SQLInjection_XSS_MixDataset.1.0.0.csv"
CSIC_PATH = PROJECT_ROOT / "csic_database.csv"
OBFUSCATION_PATH = PROJECT_ROOT / "obfuscation_dataset.xlsx"
ARTIFACT_ROOT = NOTEBOOK_DIR / "artifacts"

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

for dataset_path in (KAGGLE_PATH, CSIC_PATH, OBFUSCATION_PATH):
    if not dataset_path.exists():
        raise FileNotFoundError(f"Không tìm thấy dataset: {dataset_path}")

print("Project root:", PROJECT_ROOT)
print("TensorFlow:", tf.__version__)
print("GPU available:", bool(tf.config.list_physical_devices("GPU")))
print("Artifact root:", ARTIFACT_ROOT)


Project root: C:\Users\admin\Desktop\obfuscated-web-attack-detection
TensorFlow: 2.21.0
GPU available: False
Artifact root: C:\Users\admin\Desktop\obfuscated-web-attack-detection\cnn_lstm_parallel\artifacts


## 2. Load, làm sạch và chia riêng từng dataset

Toàn bộ bước load, chuẩn hóa, loại rỗng/trùng/xung đột và chia tập đều gọi trực tiếp từ `preprocessing/preprocess_data.py`. Notebook không viết lại loader hoặc splitter.

- `kaggle` và `csic`: stratified row split.
- `obfuscation`: group split theo `original_pattern` do module thực hiện.
- Chính sách của module giữ nguyên chữ hoa/thường và URL/HTML encoding, chỉ chuẩn hóa khoảng trắng.


In [9]:
dataset_splits, preprocessing_metadata = prep.build_dataset_splits(
    kaggle_path=str(KAGGLE_PATH),
    csic_path=str(CSIC_PATH),
    obfu_path=str(OBFUSCATION_PATH),
    test_size=TEST_SIZE,
    val_size=VAL_SIZE_WITHIN_TRAIN_VAL,
    seed=SEED,
)


In [10]:
audit_rows = []
for dataset_name, splits in dataset_splits.items():
    for split_name, frame in splits.items():
        label_counts = frame["label"].value_counts().to_dict()
        lengths = frame["payload"].str.len()
        audit_rows.append({
            "dataset": dataset_name,
            "split": split_name,
            "rows": len(frame),
            "normal": int(label_counts.get(0, 0)),
            "attack": int(label_counts.get(1, 0)),
            "attack_rate": float(frame["label"].mean()),
            "p95_length": float(lengths.quantile(0.95)),
            "over_max_len": int((lengths > MAX_LEN).sum()),
        })

dataset_audit = pd.DataFrame(audit_rows)
display(dataset_audit)
print("Tiền xử lý được thực hiện bởi preprocessing/preprocess_data.py.")


,dataset,split,rows,normal,attack,attack_rate,p95_length,over_max_len
0,kaggle,train,109197,38647,70550,0.646080,899.0,154
1,kaggle,val,12133,4294,7839,0.646089,899.4,25
2,kaggle,test,30333,10736,19597,0.646062,900.0,42
3,csic,train,8248,3348,4900,0.594083,207.0,0
4,csic,val,917,372,545,0.594329,212.0,0
5,csic,test,2292,930,1362,0.594241,206.0,0
6,obfuscation,train,217559,107998,109561,0.503592,294.0,1211
7,obfuscation,val,21982,12001,9981,0.454053,221.0,35
8,obfuscation,test,60459,30001,30458,0.503779,263.0,172


Tiền xử lý được thực hiện bởi preprocessing/preprocess_data.py.


## 3. Tokenizer và vectorization theo batch

Mỗi dataset tạo tokenizer mới và chỉ fit trên train split. Batch được pad khi cần để tránh giữ toàn bộ ma trận `N × 1024` trong RAM.

Để so sánh công bằng với mô hình tuần tự, notebook dùng cùng `padding="post"` và `truncating="post"`. Nhánh LSTM trả về toàn bộ chuỗi trạng thái rồi dùng `GlobalMaxPooling1D`, nên không phụ thuộc riêng vào final state.


In [11]:
class PayloadSequence(Sequence):
    def __init__(
        self,
        texts,
        tokenizer,
        labels=None,
        batch_size=BATCH_SIZE,
        max_len=MAX_LEN,
        shuffle=False,
        seed=SEED,
        class_weight=None,
    ):
        super().__init__()
        self.texts = np.asarray(list(texts), dtype=object)
        self.labels = None if labels is None else np.asarray(labels, dtype=np.int32)
        self.tokenizer = tokenizer
        self.batch_size = batch_size
        self.max_len = max_len
        self.shuffle = shuffle
        self.class_weight = class_weight
        self.indices = np.arange(len(self.texts))
        self.rng = np.random.default_rng(seed)
        if self.shuffle:
            self.rng.shuffle(self.indices)

    def __len__(self):
        return int(np.ceil(len(self.indices) / self.batch_size))

    def __getitem__(self, batch_index):
        start = batch_index * self.batch_size
        stop = min(start + self.batch_size, len(self.indices))
        batch_ids = self.indices[start:stop]
        sequences = self.tokenizer.texts_to_sequences(self.texts[batch_ids].tolist())
        features = pad_sequences(
            sequences,
            maxlen=self.max_len,
            padding="post",
            truncating="post",
            dtype="int32",
        )
        if self.labels is None:
            return features

        batch_labels = self.labels[batch_ids]
        if self.class_weight is None:
            return features, batch_labels

        sample_weights = np.asarray(
            [self.class_weight[int(label)] for label in batch_labels],
            dtype=np.float32,
        )
        return features, batch_labels, sample_weights

    def on_epoch_end(self):
        if self.shuffle:
            self.rng.shuffle(self.indices)


def prepare_dataset_for_training(splits):
    train_df = splits["train"]
    tokenizer = Tokenizer(
        char_level=True,
        lower=False,
        filters="",
        oov_token="<OOV>",
    )
    tokenizer.fit_on_texts(train_df["payload"])

    y_train = train_df["label"].to_numpy(dtype=np.int32)
    classes = np.unique(y_train)
    if set(classes.tolist()) != {0, 1}:
        raise ValueError(f"Train split phải có đủ nhãn 0 và 1, nhận được: {classes.tolist()}")

    weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
    class_weight = {int(label): float(weight) for label, weight in zip(classes, weights)}
    return tokenizer, len(tokenizer.word_index) + 1, class_weight


## 4. Kiến trúc CNN và LSTM song song


In [12]:
def build_parallel_cnn_lstm(dataset_name, vocab_size, max_len=MAX_LEN, embedding_dim=EMBEDDING_DIM):
    inputs = Input(shape=(max_len,), dtype="int32", name="payload_tokens")
    embedded = Embedding(
        input_dim=vocab_size,
        output_dim=embedding_dim,
        name="char_embedding",
    )(inputs)

    cnn = Conv1D(128, 3, padding="same", activation="relu", name="cnn_conv_k3")(embedded)
    cnn = MaxPooling1D(pool_size=4, name="cnn_pool_1")(cnn)
    cnn = Conv1D(128, 5, padding="same", activation="relu", name="cnn_conv_k5")(cnn)
    cnn = MaxPooling1D(pool_size=4, name="cnn_pool_2")(cnn)
    cnn_vector = GlobalMaxPooling1D(name="cnn_global_max_pool")(cnn)

    lstm_sequence = LSTM(128, return_sequences=True, name="lstm_context_sequence")(embedded)
    lstm_vector = GlobalMaxPooling1D(name="lstm_global_max_pool")(lstm_sequence)

    merged = Concatenate(name="parallel_feature_fusion")([cnn_vector, lstm_vector])
    dense = Dense(64, activation="relu", name="dense_classifier")(merged)
    dense = Dropout(0.3, name="dropout")(dense)
    outputs = Dense(1, activation="sigmoid", name="attack_probability")(dense)

    model = Model(inputs, outputs, name=f"Parallel_CNN_LSTM_{dataset_name}")
    model.compile(
        optimizer=Adam(learning_rate=1e-3),
        loss="binary_crossentropy",
        metrics=[
            "accuracy",
            AUC(name="auc"),
            Precision(name="precision"),
            Recall(name="recall"),
        ],
    )
    return model


## 5. Huấn luyện, đánh giá và lưu artifact

Mỗi dataset dùng một thư mục riêng:

```text
cnn_lstm_parallel/artifacts/
├── kaggle/
├── csic/
└── obfuscation/
```

Checkpoint và early stopping cùng theo `val_loss` như mô hình tuần tự; `patience=3` theo cấu hình thí nghiệm này. Model tốt nhất được reload trước khi đánh giá validation/test.


In [13]:
def evaluate_split(model, tokenizer, frame, split_name):
    inference_sequence = PayloadSequence(
        frame["payload"],
        tokenizer,
        labels=None,
        batch_size=BATCH_SIZE,
        max_len=MAX_LEN,
        shuffle=False,
    )
    probabilities = model.predict(inference_sequence, verbose=1).ravel()
    labels = frame["label"].to_numpy(dtype=np.int32)
    predictions = (probabilities >= DECISION_THRESHOLD).astype(np.int32)
    matrix = confusion_matrix(labels, predictions, labels=[0, 1])
    report = classification_report(
        labels,
        predictions,
        labels=[0, 1],
        target_names=["Normal (0)", "Attack (1)"],
        output_dict=True,
        zero_division=0,
    )
    result = {
        "rows": int(len(frame)),
        "accuracy": float(accuracy_score(labels, predictions)),
        "auc_roc": float(roc_auc_score(labels, probabilities)) if len(np.unique(labels)) > 1 else None,
        "false_positives": int(matrix[0, 1]),
        "false_negatives": int(matrix[1, 0]),
        "confusion_matrix": matrix.tolist(),
        "classification_report": report,
    }
    print(f"\n=== {split_name.upper()} ===")
    print(classification_report(
        labels,
        predictions,
        labels=[0, 1],
        target_names=["Normal (0)", "Attack (1)"],
        digits=4,
        zero_division=0,
    ))
    print("Confusion matrix:\n", matrix)
    print("AUC-ROC:", result["auc_roc"])
    return result


def train_one_dataset(dataset_name, splits):
    print("\n" + "=" * 80)
    print(f"TRAIN DATASET: {dataset_name.upper()}")
    print("=" * 80)
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(SEED)

    tokenizer, vocab_size, class_weight = prepare_dataset_for_training(splits)
    artifact_dir = ARTIFACT_ROOT / dataset_name
    artifact_dir.mkdir(parents=True, exist_ok=True)

    best_model_path = artifact_dir / "best_parallel_cnn_lstm.keras"
    last_model_path = artifact_dir / "last_parallel_cnn_lstm.keras"
    tokenizer_path = artifact_dir / "tokenizer.pkl"
    config_path = artifact_dir / "preprocessing_config.json"
    history_path = artifact_dir / "training_history.csv"
    results_path = artifact_dir / "metadata_and_results.json"

    train_sequence = PayloadSequence(
        splits["train"]["payload"],
        tokenizer,
        labels=splits["train"]["label"],
        shuffle=True,
        class_weight=class_weight,
    )
    val_sequence = PayloadSequence(
        splits["val"]["payload"],
        tokenizer,
        labels=splits["val"]["label"],
        shuffle=False,
    )

    model = build_parallel_cnn_lstm(dataset_name, vocab_size)
    model.summary()
    callbacks = [
        EarlyStopping(
            monitor="val_loss",
            mode="min",
            patience=3,
            restore_best_weights=True,
            verbose=1,
        ),
        ModelCheckpoint(
            filepath=str(best_model_path),
            monitor="val_loss",
            mode="min",
            save_best_only=True,
            verbose=1,
        ),
    ]

    started = time.perf_counter()
    history = model.fit(
        train_sequence,
        validation_data=val_sequence,
        epochs=EPOCHS,
        callbacks=callbacks,
        verbose=1,
    )
    training_seconds = time.perf_counter() - started
    model.save(last_model_path)
    with tokenizer_path.open("wb") as file:
        pickle.dump(tokenizer, file)

    best_model = tf.keras.models.load_model(best_model_path)
    validation_result = evaluate_split(best_model, tokenizer, splits["val"], f"{dataset_name} validation")
    test_result = evaluate_split(best_model, tokenizer, splits["test"], f"{dataset_name} test")

    preprocessing_config = {
        "dataset": dataset_name,
        "seed": SEED,
        "preprocessing_source": "preprocessing/preprocess_data.py",
        "split_target": {"train": 0.72, "validation": 0.08, "test": 0.20},
        "group_column": "original_pattern" if dataset_name == "obfuscation" else None,
        "csic_input_contract": (
            preprocessing_metadata["preprocessing_policy"]["csic_payload_policy"]
            if dataset_name == "csic"
            else None
        ),
        "url_decode": preprocessing_metadata["preprocessing_policy"]["url_decode"],
        "html_unescape": preprocessing_metadata["preprocessing_policy"]["html_unescape"],
        "lowercase": preprocessing_metadata["preprocessing_policy"]["lowercase"],
        "whitespace_normalization_only": preprocessing_metadata["preprocessing_policy"]["whitespace_normalization_only"],
        "tokenizer_fit_split": "train",
        "vocab_size": int(vocab_size),
        "max_len": MAX_LEN,
        "padding": "post",
        "truncating": "post",
        "decision_threshold": DECISION_THRESHOLD,
        "class_weight": {str(key): float(value) for key, value in class_weight.items()},
        "splits": {name: prep.summarize(frame) for name, frame in splits.items()},
    }
    with config_path.open("w", encoding="utf-8") as file:
        json.dump(preprocessing_config, file, ensure_ascii=False, indent=2)

    pd.DataFrame(history.history).to_csv(history_path, index=False, encoding="utf-8")
    result = {
        "dataset": dataset_name,
        "artifact_dir": str(artifact_dir),
        "preprocessing": preprocessing_config,
        "model": {
            "name": best_model.name,
            "parameter_count": int(best_model.count_params()),
            "topology": "Shared Embedding -> [CNN branch -> GlobalMaxPool || LSTM(return_sequences=True) -> GlobalMaxPool] -> Concatenate -> Dense -> Sigmoid",
            "embedding_dim": EMBEDDING_DIM,
            "cnn_filters": [128, 128],
            "cnn_kernel_sizes": [3, 5],
            "cnn_pool_sizes": [4, 4],
            "lstm_units": 128,
            "lstm_return_sequences": True,
            "lstm_pooling": "GlobalMaxPooling1D",
            "dense_units": 64,
            "dropout": 0.3,
            "learning_rate": 1e-3,
            "batch_size": BATCH_SIZE,
            "epochs_requested": EPOCHS,
            "epochs_ran": int(len(history.history["loss"])),
            "early_stopping_monitor": "val_loss",
            "early_stopping_patience": 3,
        },
        "training_seconds": float(training_seconds),
        "training_history": {
            key: [float(value) for value in values]
            for key, values in history.history.items()
        },
        "evaluation": {
            "validation": validation_result,
            "test": test_result,
        },
    }
    with results_path.open("w", encoding="utf-8") as file:
        json.dump(result, file, ensure_ascii=False, indent=2)

    del model, best_model, history, train_sequence, val_sequence
    gc.collect()
    tf.keras.backend.clear_session()
    print("Đã lưu kết quả:", results_path)
    return result


## 6. Chọn dataset và chạy huấn luyện

Mặc định notebook train lần lượt cả ba dataset. Để chạy lại riêng một nguồn, ví dụ CSIC, đổi thành:

```python
DATASETS_TO_TRAIN = ["csic"]
```

Kết quả đã hoàn tất của từng dataset được lưu ngay trước khi chuyển sang dataset tiếp theo.


In [14]:
DATASETS_TO_TRAIN = ["kaggle", "csic", "obfuscation"]

unknown_datasets = set(DATASETS_TO_TRAIN) - set(dataset_splits)
if unknown_datasets:
    raise ValueError(f"Dataset không hợp lệ: {sorted(unknown_datasets)}")

all_results = {}
all_results_path = ARTIFACT_ROOT / "all_dataset_results.json"

for dataset_name in DATASETS_TO_TRAIN:
    all_results[dataset_name] = train_one_dataset(dataset_name, dataset_splits[dataset_name])
    ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
    with all_results_path.open("w", encoding="utf-8") as file:
        json.dump(all_results, file, ensure_ascii=False, indent=2)

comparison_rows = []
for dataset_name, result in all_results.items():
    test = result["evaluation"]["test"]
    attack = test["classification_report"]["Attack (1)"]
    comparison_rows.append({
        "dataset": dataset_name,
        "test_rows": test["rows"],
        "accuracy": test["accuracy"],
        "auc_roc": test["auc_roc"],
        "attack_precision": attack["precision"],
        "attack_recall": attack["recall"],
        "attack_f1": attack["f1-score"],
        "false_positives": test["false_positives"],
        "false_negatives": test["false_negatives"],
        "training_seconds": result["training_seconds"],
    })

comparison = pd.DataFrame(comparison_rows)
comparison.to_csv(ARTIFACT_ROOT / "dataset_comparison.csv", index=False, encoding="utf-8")
display(comparison)
print("Đã lưu tổng hợp:", all_results_path)



TRAIN DATASET: KAGGLE


Model: "Parallel_CNN_LSTM_kaggle"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ payload_tokens      │ (None, 1024)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ char_embedding      │ (None, 1024, 64)  │     13,568 │ payload_tokens[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cnn_conv_k3         │ (None, 1024, 128) │     24,704 │ char_embedding[0… │
│ (Conv1D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cnn_pool_1          │ (None, 256, 128)  │          0 │ cnn_conv_k3[0][0] │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cnn_conv_k5         │ (None, 256, 128)  │     82,048 │ cnn_pool_1[0][0]  │
│ (Conv1D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cnn_pool_2          │ (None, 64, 128)   │          0 │ cnn_conv_k5[0][0] │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_context_seque… │ (None, 1024, 128) │     98,816 │ char_embedding[0… │
│ (LSTM)              │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cnn_global_max_pool │ (None, 128)       │          0 │ cnn_pool_2[0][0]  │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_global_max_po… │ (None, 128)       │          0 │ lstm_context_seq… │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ parallel_feature_f… │ (None, 256)       │          0 │ cnn_global_max_p… │
│ (Concatenate)       │                   │            │ lstm_global_max_… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_classifier    │ (None, 64)        │     16,448 │ parallel_feature… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 64)        │          0 │ dense_classifier… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attack_probability  │ (None, 1)         │         65 │ dropout[0][0]     │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 235,649 (920.50 KB)

 Trainable params: 235,649 (920.50 KB)

...

 Total params: 233,537 (912.25 KB)

 Trainable params: 233,537 (912.25 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
1700/1700 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.9810 - auc: 0.9961 - loss: 0.0432 - precision: 0.9948 - recall: 0.9653
Epoch 1: val_loss improved from None to 0.00001, saving model to C:\Users\admin\Desktop\obfuscated-web-attack-detection\cnn_lstm_parallel\artifacts\obfuscation\best_parallel_cnn_lstm.keras

Epoch 1: finished saving model to C:\Users\admin\Desktop\obfuscated-web-attack-detection\cnn_lstm_parallel\artifacts\obfuscation\best_parallel_cnn_lstm.keras
1700/1700 ━━━━━━━━━━━━━━━━━━━━ 5250s 3s/step - accuracy: 0.9964 - auc: 0.9999 - loss: 0.0091 - precision: 0.9991 - recall: 0.9937 - val_accuracy: 1.0000 - val_auc: 1.0000 - val_loss: 7.0982e-06 - val_precision: 1.0000 - val_recall: 1.0000
Epoch 2/50
1700/1700 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 1.0000 - auc: 1.0000 - loss: 2.5214e-05 - precision: 1.0000 - recall: 1.0000
Epoch 2: val_loss did not improve from 0.00001
1700/1700 ━━━━━━━━━━━━━━━━━━━━ 5107s 3s/step - accuracy: 0.9999 - auc: 1.0000 - loss: 

KeyboardInterrupt: 

### 7.1. Khôi phục JSON khi quá trình train bị interrupt

Nếu đã có `best_parallel_cnn_lstm.keras` nhưng chưa có `metadata_and_results.json`, cell này tạo lại tokenizer từ đúng train split, đọc lịch sử các epoch hoàn chỉnh từ output đã lưu của cell train, đánh giá checkpoint trên validation/test và ghi JSON mà không train lại. Epoch đang chạy dở tại thời điểm interrupt sẽ bị loại khỏi lịch sử.


In [ ]:
# Chỉ cần chạy cell này sau khi đã chạy các cell setup/preprocessing/helper phía trên.
RECOVER_DATASETS = ["obfuscation"]

import re


def recover_training_history_from_notebook(dataset_name):
    notebook_file = NOTEBOOK_DIR / "cnn_lstm_parallel_experiment.ipynb"
    if not notebook_file.exists():
        return {}, [], []

    with notebook_file.open("r", encoding="utf-8") as file:
        saved_notebook = json.load(file)

    try:
        train_cell = next(
            cell for cell in saved_notebook["cells"]
            if cell.get("id") == "run-code"
        )
    except StopIteration:
        return {}, [], []

    stdout = "\n".join(
        "".join(output.get("text", []))
        for output in train_cell.get("outputs", [])
        if output.get("output_type") == "stream" and output.get("name") == "stdout"
    )
    stdout = re.sub(r"\x1b\[[0-?]*[ -/]*[@-~]", "", stdout)

    marker = f"TRAIN DATASET: {dataset_name.upper()}"
    marker_position = stdout.find(marker)
    if marker_position < 0:
        return {}, [], []

    dataset_log = stdout[marker_position:]
    next_marker = dataset_log.find("TRAIN DATASET:", len(marker))
    if next_marker >= 0:
        dataset_log = dataset_log[:next_marker]

    metric_names = [
        "accuracy", "auc", "loss", "precision", "recall",
        "val_accuracy", "val_auc", "val_loss", "val_precision", "val_recall",
    ]
    history = {name: [] for name in metric_names}
    completed_epoch_seconds = []
    started_epochs = []
    metric_pattern = re.compile(
        r"(?P<name>val_accuracy|val_auc|val_loss|val_precision|val_recall|"
        r"accuracy|auc|loss|precision|recall): "
        r"(?P<value>[0-9.eE+-]+)"
    )
    epoch_pattern = re.compile(
        r"Epoch\s+(?P<epoch>\d+)/\d+\s*(?P<body>.*?)"
        r"(?=Epoch\s+\d+/\d+|\Z)",
        re.DOTALL,
    )

    for epoch_match in epoch_pattern.finditer(dataset_log):
        epoch_number = int(epoch_match.group("epoch"))
        started_epochs.append(epoch_number)
        completed_lines = [
            line for line in epoch_match.group("body").splitlines()
            if "val_accuracy:" in line and "val_loss:" in line
        ]
        if not completed_lines:
            continue

        summary_line = completed_lines[-1]
        values = {
            match.group("name"): float(match.group("value"))
            for match in metric_pattern.finditer(summary_line)
        }
        if not all(name in values for name in metric_names):
            continue

        for name in metric_names:
            history[name].append(values[name])

        seconds_match = re.search(r"\b(\d+)s\b", summary_line)
        if seconds_match:
            completed_epoch_seconds.append(int(seconds_match.group(1)))

    if not history["loss"]:
        return {}, [], started_epochs
    return history, completed_epoch_seconds, started_epochs


def recover_interrupted_result(dataset_name):
    if dataset_name not in dataset_splits:
        raise ValueError(f"Dataset không hợp lệ: {dataset_name!r}")

    splits = dataset_splits[dataset_name]
    artifact_dir = ARTIFACT_ROOT / dataset_name
    best_model_path = artifact_dir / "best_parallel_cnn_lstm.keras"
    tokenizer_path = artifact_dir / "tokenizer.pkl"
    config_path = artifact_dir / "preprocessing_config.json"
    history_path = artifact_dir / "training_history.csv"
    results_path = artifact_dir / "metadata_and_results.json"

    if results_path.exists():
        print(f"{dataset_name.upper()}: JSON đã tồn tại, không ghi đè: {results_path}")
        with results_path.open("r", encoding="utf-8") as file:
            return json.load(file)

    if not best_model_path.exists():
        raise FileNotFoundError(
            f"Không có checkpoint để khôi phục {dataset_name!r}: {best_model_path}"
        )

    recovered_history, completed_epoch_seconds, started_epochs = (
        recover_training_history_from_notebook(dataset_name)
    )
    completed_epochs = len(recovered_history.get("loss", []))
    if completed_epochs:
        print(
            f"{dataset_name.upper()}: khôi phục lịch sử của "
            f"{completed_epochs} epoch hoàn chỉnh từ output notebook."
        )
    else:
        print(
            f"Cảnh báo: không đọc được lịch sử epoch của {dataset_name.upper()} "
            "từ output notebook."
        )

    tokenizer, vocab_size, class_weight = prepare_dataset_for_training(splits)
    best_model = tf.keras.models.load_model(best_model_path)

    checkpoint_vocab_size = int(best_model.get_layer("char_embedding").input_dim)
    if checkpoint_vocab_size != int(vocab_size):
        raise ValueError(
            "Tokenizer tạo lại không khớp checkpoint: "
            f"checkpoint vocab={checkpoint_vocab_size}, current vocab={vocab_size}. "
            "Không tiếp tục để tránh đánh giá sai token mapping."
        )

    with tokenizer_path.open("wb") as file:
        pickle.dump(tokenizer, file)

    validation_result = evaluate_split(
        best_model,
        tokenizer,
        splits["val"],
        f"{dataset_name} validation (recovered)",
    )
    test_result = evaluate_split(
        best_model,
        tokenizer,
        splits["test"],
        f"{dataset_name} test (recovered)",
    )

    preprocessing_config = {
        "dataset": dataset_name,
        "seed": SEED,
        "preprocessing_source": "preprocessing/preprocess_data.py",
        "split_target": {"train": 0.72, "validation": 0.08, "test": 0.20},
        "group_column": "original_pattern" if dataset_name == "obfuscation" else None,
        "csic_input_contract": (
            preprocessing_metadata["preprocessing_policy"]["csic_payload_policy"]
            if dataset_name == "csic"
            else None
        ),
        "url_decode": preprocessing_metadata["preprocessing_policy"]["url_decode"],
        "html_unescape": preprocessing_metadata["preprocessing_policy"]["html_unescape"],
        "lowercase": preprocessing_metadata["preprocessing_policy"]["lowercase"],
        "whitespace_normalization_only": preprocessing_metadata["preprocessing_policy"]["whitespace_normalization_only"],
        "tokenizer_fit_split": "train",
        "vocab_size": int(vocab_size),
        "max_len": MAX_LEN,
        "padding": "post",
        "truncating": "post",
        "decision_threshold": DECISION_THRESHOLD,
        "class_weight": {str(key): float(value) for key, value in class_weight.items()},
        "splits": {name: prep.summarize(frame) for name, frame in splits.items()},
    }
    with config_path.open("w", encoding="utf-8") as file:
        json.dump(preprocessing_config, file, ensure_ascii=False, indent=2)

    if recovered_history:
        pd.DataFrame(recovered_history).to_csv(
            history_path,
            index=False,
            encoding="utf-8",
        )

    result = {
        "dataset": dataset_name,
        "artifact_dir": str(artifact_dir),
        "preprocessing": preprocessing_config,
        "model": {
            "name": best_model.name,
            "parameter_count": int(best_model.count_params()),
            "topology": "Shared Embedding -> [CNN branch -> GlobalMaxPool || LSTM(return_sequences=True) -> GlobalMaxPool] -> Concatenate -> Dense -> Sigmoid",
            "embedding_dim": EMBEDDING_DIM,
            "cnn_filters": [128, 128],
            "cnn_kernel_sizes": [3, 5],
            "cnn_pool_sizes": [4, 4],
            "lstm_units": 128,
            "lstm_return_sequences": True,
            "lstm_pooling": "GlobalMaxPooling1D",
            "dense_units": 64,
            "dropout": 0.3,
            "learning_rate": 1e-3,
            "batch_size": BATCH_SIZE,
            "epochs_requested": EPOCHS,
            "epochs_ran": completed_epochs if completed_epochs else None,
            "early_stopping_monitor": "val_loss",
            "early_stopping_patience": 3,
        },
        "training_seconds": (
            float(sum(completed_epoch_seconds))
            if len(completed_epoch_seconds) == completed_epochs and completed_epochs
            else None
        ),
        "training_history": recovered_history,
        "recovery": {
            "recovered_from_checkpoint": str(best_model_path),
            "training_was_interrupted": True,
            "training_history_available": bool(recovered_history),
            "training_history_source": str(NOTEBOOK_DIR / "cnn_lstm_parallel_experiment.ipynb"),
            "completed_epochs_recovered": completed_epochs,
            "started_epochs_seen": started_epochs,
            "incomplete_epochs_excluded": [
                epoch for epoch in started_epochs if epoch > completed_epochs
            ],
            "completed_epoch_seconds": completed_epoch_seconds,
            "note": "Lịch sử được parse từ output đã lưu; checkpoint là model tốt nhất theo val_loss.",
        },
        "evaluation": {
            "validation": validation_result,
            "test": test_result,
        },
    }
    with results_path.open("w", encoding="utf-8") as file:
        json.dump(result, file, ensure_ascii=False, indent=2)

    del best_model
    gc.collect()
    tf.keras.backend.clear_session()
    print(f"{dataset_name.upper()}: đã khôi phục JSON tại {results_path}")
    return result


recovered_results = {
    dataset_name: recover_interrupted_result(dataset_name)
    for dataset_name in RECOVER_DATASETS
}

# Đồng bộ biến tổng hợp nếu cell train trước đó đã tạo biến all_results.
if "all_results" not in globals():
    all_results = {}
all_results.update(recovered_results)
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
with (ARTIFACT_ROOT / "all_dataset_results.json").open("w", encoding="utf-8") as file:
    json.dump(all_results, file, ensure_ascii=False, indent=2)


## 7. Nguyên tắc đọc kết quả

- Ba test set thuộc ba phân phối khác nhau, vì vậy không kết luận dataset/model “tốt hơn” chỉ từ accuracy.
- Luôn xem Attack Recall, Attack F1, false negative, false positive và AUC-ROC.
- Split obfuscation đo khả năng khái quát sang `original_pattern` chưa thấy trong train, nên khó hơn row split nhưng an toàn hơn về leakage.
- So sánh topology chỉ hợp lệ khi CNN-only, CNN-LSTM tuần tự và CNN-LSTM song song dùng cùng dataset split, tokenizer rule, `MAX_LEN`, threshold và seed.


## 8. Thống kê cuối notebook cho từng dataset

Hai bảng dưới đây tổng hợp quy mô, phân bố nhãn, độ dài, tỷ lệ vượt `MAX_LEN` của từng split và bổ sung test metrics sau khi dataset tương ứng đã được train. Các bảng cũng được lưu trong `cnn_lstm_parallel/artifacts/`.


In [15]:
split_statistics_rows = []
dataset_statistics_rows = []
trained_results = globals().get("all_results", {})

for dataset_name, splits in dataset_splits.items():
    clean_summary = preprocessing_metadata["datasets"][dataset_name]["clean"]
    total_over_max_len = 0

    for split_name, frame in splits.items():
        lengths = frame["payload"].str.len()
        label_counts = frame["label"].value_counts().to_dict()
        over_max_len = int((lengths > MAX_LEN).sum())
        total_over_max_len += over_max_len
        split_statistics_rows.append({
            "dataset": dataset_name,
            "split": split_name,
            "rows": int(len(frame)),
            "normal": int(label_counts.get(0, 0)),
            "attack": int(label_counts.get(1, 0)),
            "attack_rate": float(frame["label"].mean()),
            "mean_length": float(lengths.mean()),
            "median_length": float(lengths.median()),
            "p95_length": float(lengths.quantile(0.95)),
            "max_length": int(lengths.max()),
            "over_max_len": over_max_len,
            "over_max_len_rate": float((lengths > MAX_LEN).mean()),
        })

    clean_rows = int(clean_summary["rows"])
    clean_labels = clean_summary["label_counts"]
    overview = {
        "dataset": dataset_name,
        "clean_rows": clean_rows,
        "normal": int(clean_labels.get("0", 0)),
        "attack": int(clean_labels.get("1", 0)),
        "attack_rate": int(clean_labels.get("1", 0)) / clean_rows if clean_rows else 0.0,
        "train_rows": int(len(splits["train"])),
        "validation_rows": int(len(splits["val"])),
        "test_rows": int(len(splits["test"])),
        "p95_length": float(clean_summary["length"]["p95"]),
        "max_length": int(clean_summary["length"]["max"]),
        "over_max_len": total_over_max_len,
        "over_max_len_rate": total_over_max_len / clean_rows if clean_rows else 0.0,
        "test_accuracy": None,
        "test_auc_roc": None,
        "test_attack_recall": None,
        "test_attack_f1": None,
        "test_false_positives": None,
        "test_false_negatives": None,
    }

    if dataset_name in trained_results:
        test_result = trained_results[dataset_name]["evaluation"]["test"]
        attack_report = test_result["classification_report"]["Attack (1)"]
        overview.update({
            "test_accuracy": float(test_result["accuracy"]),
            "test_auc_roc": test_result["auc_roc"],
            "test_attack_recall": float(attack_report["recall"]),
            "test_attack_f1": float(attack_report["f1-score"]),
            "test_false_positives": int(test_result["false_positives"]),
            "test_false_negatives": int(test_result["false_negatives"]),
        })

    dataset_statistics_rows.append(overview)

dataset_statistics = pd.DataFrame(dataset_statistics_rows)
split_statistics = pd.DataFrame(split_statistics_rows)

ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
dataset_statistics.to_csv(ARTIFACT_ROOT / "dataset_statistics.csv", index=False, encoding="utf-8")
split_statistics.to_csv(ARTIFACT_ROOT / "dataset_split_statistics.csv", index=False, encoding="utf-8")

print("=== TỔNG QUAN TỪNG DATASET ===")
display(dataset_statistics)
print("=== CHI TIẾT TỪNG SPLIT ===")
display(split_statistics)
print("Đã lưu:", ARTIFACT_ROOT / "dataset_statistics.csv")
print("Đã lưu:", ARTIFACT_ROOT / "dataset_split_statistics.csv")


=== TỔNG QUAN TỪNG DATASET ===


,dataset,clean_rows,normal,attack,attack_rate,train_rows,validation_rows,test_rows,p95_length,max_length,over_max_len,over_max_len_rate,test_accuracy,test_auc_roc,test_attack_recall,test_attack_f1,test_false_positives,test_false_negatives
0,kaggle,151663,53677,97986,0.646077,109197,12133,30333,899.0,8493,221,0.001457,0.998187,0.999976,0.997959,0.998596,15.0,40.0
1,csic,11457,4650,6807,0.594135,8248,917,2292,207.0,752,0,0.000000,0.785777,0.844752,0.710720,0.797693,97.0,394.0
2,obfuscation,300000,150000,150000,0.500000,217559,21982,60459,279.0,6853,1418,0.004727,NaN,NaN,NaN,NaN,NaN,NaN


=== CHI TIẾT TỪNG SPLIT ===


,dataset,split,rows,normal,attack,attack_rate,mean_length,median_length,p95_length,max_length,over_max_len,over_max_len_rate
0,kaggle,train,109197,38647,70550,0.646080,356.449097,274.0,899.0,8493,154,0.001410
1,kaggle,val,12133,4294,7839,0.646089,356.005192,269.0,899.4,5909,25,0.002060
2,kaggle,test,30333,10736,19597,0.646062,356.545709,273.0,900.0,3070,42,0.001385
3,csic,train,8248,3348,4900,0.594083,100.864088,55.0,207.0,752,0,0.000000
4,csic,val,917,372,545,0.594329,101.118866,52.0,212.0,740,0,0.000000
5,csic,test,2292,930,1362,0.594241,101.555410,56.0,206.0,740,0,0.000000
6,obfuscation,train,217559,107998,109561,0.503592,114.505045,77.0,294.0,6853,1211,0.005566
7,obfuscation,val,21982,12001,9981,0.454053,94.127468,74.0,221.0,3276,35,0.001592
8,obfuscation,test,60459,30001,30458,0.503779,107.848195,76.0,263.0,3969,172,0.002845


Đã lưu: C:\Users\admin\Desktop\obfuscated-web-attack-detection\cnn_lstm_parallel\artifacts\dataset_statistics.csv
Đã lưu: C:\Users\admin\Desktop\obfuscated-web-attack-detection\cnn_lstm_parallel\artifacts\dataset_split_statistics.csv


## 9. Biểu đồ kết quả CNN ∥ LSTM song song theo từng dataset

Cell dưới đây chỉ đọc artifact của kiến trúc **CNN ∥ LSTM song song**. Mỗi nhóm trên trục ngang là một dataset; ba cột biểu diễn Precision, Recall và F1-score của lớp `Attack (1)` trên tập test tại ngưỡng mặc định 0.5.


In [ ]:
PLOT_DATASETS = ["kaggle", "csic", "obfuscation"]

import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

VALID_PLOT_DATASETS = {"kaggle", "csic", "obfuscation"}
unknown_datasets = set(PLOT_DATASETS) - VALID_PLOT_DATASETS
if unknown_datasets:
    raise ValueError(f"Dataset không hợp lệ: {sorted(unknown_datasets)}")


def load_parallel_attack_metrics(dataset_name):
    result_path = ARTIFACT_ROOT / dataset_name / "metadata_and_results.json"
    if not result_path.exists():
        raise FileNotFoundError(
            f"Chưa có artifact CNN ∥ LSTM song song cho dataset {dataset_name!r}: "
            f"{result_path}\nHãy train dataset này bằng notebook rồi chạy lại cell biểu đồ."
        )

    with result_path.open("r", encoding="utf-8") as file:
        payload = json.load(file)

    if payload.get("dataset") != dataset_name:
        raise ValueError(
            f"Artifact {result_path} không khớp dataset: "
            f"metadata={payload.get('dataset')!r}, expected={dataset_name!r}."
        )

    try:
        attack_report = payload["evaluation"]["test"]["classification_report"]["Attack (1)"]
    except KeyError as exc:
        raise KeyError(
            f"Artifact {result_path} thiếu kết quả lớp 'Attack (1)': {exc}"
        ) from exc

    metrics = [
        100.0 * float(attack_report["precision"]),
        100.0 * float(attack_report["recall"]),
        100.0 * float(attack_report["f1-score"]),
    ]
    return metrics, result_path


metric_names = ["Precision", "Recall", "F1-score"]
metric_rows = []
plotted_datasets = []
missing_datasets = []
result_paths = {}

for dataset_name in PLOT_DATASETS:
    try:
        metrics, result_path = load_parallel_attack_metrics(dataset_name)
    except FileNotFoundError:
        missing_datasets.append(dataset_name)
        continue

    metric_rows.append(metrics)
    plotted_datasets.append(dataset_name)
    result_paths[dataset_name] = result_path

if not metric_rows:
    raise FileNotFoundError(
        "Chưa có artifact CNN ∥ LSTM song song cho bất kỳ dataset nào trong "
        f"{PLOT_DATASETS}. Hãy train ít nhất một dataset trước."
    )

if missing_datasets:
    print(
        "Cảnh báo: bỏ qua dataset chưa có artifact:",
        ", ".join(name.upper() for name in missing_datasets),
    )

metric_values = np.asarray(metric_rows, dtype=float)
dataset_labels = [name.upper() for name in plotted_datasets]

metric_table = pd.DataFrame(
    metric_values,
    index=dataset_labels,
    columns=metric_names,
)
metric_table.index.name = "Dataset"
display(metric_table.round(2).style.format("{:.2f}%"))

x = np.arange(len(plotted_datasets))
width = 0.22
colors = ["#5B9BD5", "#ED7D31", "#A5A5A5"]

fig, ax = plt.subplots(figsize=(10, 6))
for metric_index, (metric_name, color) in enumerate(zip(metric_names, colors)):
    offset = (metric_index - 1) * width
    bars = ax.bar(
        x + offset,
        metric_values[:, metric_index],
        width,
        label=metric_name.upper(),
        color=color,
    )
    ax.bar_label(bars, fmt="%.2f", padding=3, fontsize=9)

lowest_metric = float(metric_values.min())
y_min = max(0.0, np.floor((lowest_metric - 5.0) / 5.0) * 5.0)
ax.set_ylim(y_min, 100.5)
ax.set_ylabel("Giá trị (%)")
ax.set_xticks(x)
ax.set_xticklabels(dataset_labels)
ax.set_title("CNN ∥ LSTM SONG SONG TRÊN TỪNG DATASET", fontsize=16, pad=18)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.10), ncol=3, frameon=False)
ax.grid(axis="y", alpha=0.30)
ax.set_axisbelow(True)
for spine in ("top", "right"):
    ax.spines[spine].set_visible(False)

fig.tight_layout()
chart_path = ARTIFACT_ROOT / "parallel_metrics_by_dataset.png"
fig.savefig(chart_path, dpi=300, bbox_inches="tight")
plt.show()

for dataset_name, result_path in result_paths.items():
    print(f"{dataset_name.upper():12s}: {result_path}")
print("Đã lưu biểu đồ:", chart_path)
